# Project LLM Fine-Tuning (LoRA)

This notebook fine-tunes Qwen2-0.5B or TinyLlama on the project management dataset (summary, features, roles, goals, timeline, backlog) using LoRA. Designed for Windows RTX 4050 (6GB VRAM) — uses fp16, no bitsandbytes required.

In [ ]:
# 1. Imports & config
import sys
from pathlib import Path

# Find AI root (contains llms/fine_tune/prepare_dataset.py)
_cwd = Path.cwd()
_candidates = [_cwd, _cwd.parent, _cwd / "AI"]
_ai_root = None
for c in _candidates:
    p = c / "llms" / "fine_tune" / "prepare_dataset.py"
    if p.exists():
        _ai_root = c if (c / "llms").exists() else c.parent
        break
if _ai_root is None:
    _ai_root = _cwd / "AI" if (_cwd / "AI").exists() else _cwd
if str(_ai_root) not in sys.path:
    sys.path.insert(0, str(_ai_root))
_nb_dir = _ai_root / "llms" / "fine_tune"  # notebook lives here

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType
from datasets import load_from_disk

# Config
MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"  # or TinyLlama/TinyLlama-1.1B-Chat-v1.0
DATASET_DIR = _nb_dir / "dataset"
TOKENIZED_DIR = _nb_dir / "tokenized"
OUTPUT_DIR = _nb_dir / "qwen_project_manager_lora"  # adapter output
MAX_LENGTH = 512
BATCH_SIZE = 2  # Reduce if OOM on 6GB
EPOCHS = 3

## 2. Prepare & load dataset

Run `prepare_dataset.py` first if tokenized data does not exist, or load from disk.

In [ ]:
# Prepare dataset (run once; then can load from disk)
from llms.fine_tune.prepare_dataset import prepare_tokenized_dataset, load_all_sections

# Check raw examples
examples = load_all_sections()
print(f"Loaded {len(examples)} examples")
if examples:
    print("Sample prompt (first 200 chars):", examples[0]["prompt"][:200])
    print("Sample response:", examples[0]["response"][:150])

# Tokenize (or load from disk if already prepared)
tokenized_path = TOKENIZED_DIR / "tokenized_project_management_qwen"
if tokenized_path.exists():
    dataset = load_from_disk(str(tokenized_path))
    print(f"Loaded tokenized dataset from {tokenized_path}")
else:
    dataset = prepare_tokenized_dataset(
        model_name="qwen",
        max_length=MAX_LENGTH,
        output_dir=str(tokenized_path),
    )
print(f"Dataset size: {len(dataset)}")

## 3. Load base model & apply LoRA

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Load base model (fp16 for 6GB VRAM)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
if torch.cuda.is_available():
    model = model.to("cuda")

# LoRA config
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 4. Train

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    logging_dir=str(OUTPUT_DIR / "logs"),
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

trainer.train()

## 5. Save adapter

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f"Saved adapter and tokenizer to {OUTPUT_DIR}")
print("Set PEFT_ADAPTER_PATH in AI/.env to use this adapter:")
print(f"  PEFT_ADAPTER_PATH=llms/fine_tune/qwen_project_manager_lora")

## 6. Quick inference test

In [ ]:
from peft import PeftModel

# Reload base + adapter for clean inference
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model_infer = PeftModel.from_pretrained(base_model, str(OUTPUT_DIR))
if torch.cuda.is_available():
    model_infer = model_infer.to("cuda")

# Test with a short prompt
test_prompt = "You are a senior AI assistant. Read the proposal and generate a summary.\n\n<<<PROPOSAL>>>\nBuild a task management web app for small teams.\n<<<END PROPOSAL>>>\n\nsummary:"
inputs = tokenizer(test_prompt, return_tensors="pt")
if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}
outputs = model_infer.generate(**inputs, max_new_tokens=64, do_sample=True, temperature=0.4)
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Generated response:")
print(response.strip())